In [0]:
-- Pergunta 1: Como a quantidade de passageiros transportados evoluiu ao longo dos anos?

SELECT
    ANO,
    SUM(COALESCE(PASSAGEIROS_PAGOS, 0) + COALESCE(PASSAGEIROS_GRATIS, 0)) AS total_passageiros,
    SUM(COALESCE(PASSAGEIROS_PAGOS, 0)) AS passageiros_pagos,
    SUM(COALESCE(PASSAGEIROS_GRATIS, 0)) AS passageiros_gratis
FROM workspace.default.vw_voos_completos
GROUP BY ANO
ORDER BY ANO;

ANO,total_passageiros,passageiros_pagos,passageiros_gratis
2000,38648361,37413460,1234901
2001,40107695,38545299,1562396
2002,40008782,38301313,1707469
2003,38188226,37203867,984359
2004,42200355,41174131,1026224
2005,50408315,49106864,1301451
2006,55035245,53863577,1171668
2007,60775826,59611480,1164346
2008,64792452,63378804,1413648
2009,71203924,69555497,1648427


In [0]:
-- Pergunta 2: Quais foram as rotas com maior movimentação de passageiros ao longo do período analisado?

SELECT
    AEROPORTO_ORIGEM_NOME,
    AEROPORTO_ORIGEM_PAIS,
    AEROPORTO_DESTINO_NOME,
    AEROPORTO_DESTINO_PAIS,
    SUM(DECOLAGENS) AS quantidade_voos,
    SUM(COALESCE(PASSAGEIROS_PAGOS, 0) + COALESCE(PASSAGEIROS_GRATIS, 0)) AS total_passageiros
FROM workspace.default.vw_voos_completos
WHERE AEROPORTO_ORIGEM_NOME IS NOT NULL
  AND AEROPORTO_DESTINO_NOME IS NOT NULL
GROUP BY AEROPORTO_ORIGEM_NOME, AEROPORTO_ORIGEM_PAIS, AEROPORTO_DESTINO_NOME, AEROPORTO_DESTINO_PAIS
ORDER BY total_passageiros DESC
LIMIT 20;

AEROPORTO_ORIGEM_NOME,AEROPORTO_ORIGEM_PAIS,AEROPORTO_DESTINO_NOME,AEROPORTO_DESTINO_PAIS,quantidade_voos,total_passageiros
SÃO PAULO,BRASIL,RIO DE JANEIRO,BRASIL,587452,53949158
RIO DE JANEIRO,BRASIL,SÃO PAULO,BRASIL,583384,53790305
SÃO PAULO,BRASIL,BRASÍLIA,BRASIL,210743,23070152
BRASÍLIA,BRASIL,SÃO PAULO,BRASIL,209173,22848154
RIO DE JANEIRO,BRASIL,GUARULHOS,BRASIL,233825,19475429
BRASÍLIA,BRASIL,RIO DE JANEIRO,BRASIL,193313,19454080
RIO DE JANEIRO,BRASIL,BRASÍLIA,BRASIL,191827,19417812
GUARULHOS,BRASIL,RIO DE JANEIRO,BRASIL,234433,19027414
SALVADOR,BRASIL,GUARULHOS,BRASIL,153662,18966973
GUARULHOS,BRASIL,SALVADOR,BRASIL,158116,18775088


In [0]:
-- Pergunta 3: Quais empresas aéreas apresentaram a maior média de passageiros por decolagem?
-- A coluna DECOLAGENS contém a quantidade real de decolagens realizadas na etapa de voo.

SELECT
    EMPRESA_NOME,
    SUM(COALESCE(PASSAGEIROS_PAGOS, 0) + COALESCE(PASSAGEIROS_GRATIS, 0)) AS total_passageiros,
    SUM(DECOLAGENS) AS total_decolagens,
    ROUND(
        SUM(COALESCE(PASSAGEIROS_PAGOS, 0) + COALESCE(PASSAGEIROS_GRATIS, 0)) 
        / NULLIF(SUM(DECOLAGENS), 0), 2
    ) AS media_passageiros_por_decolagem
FROM workspace.default.vw_voos_completos
GROUP BY EMPRESA_NOME
ORDER BY media_passageiros_por_decolagem DESC
LIMIT 20;

EMPRESA_NOME,total_passageiros,total_decolagens,media_passageiros_por_decolagem
WHITEJETS TRANSPORTES AÉREOS S.A.,21066,39,540.15
KLM CIA. REAL HOLANDESA DE AVIAÇÃO,7801026,27887,279.74
NORWEGIAN AIR UK LIMITED,107587,402,267.63
JAPAN AIRLINES INTERNATIONAL COMPANY LIMITED,1022354,3921,260.74
SOCIÉTÉ AIR FRANCE,15488703,59859,258.75
ITALIA TRANSPORTO AEREO S.P.A.,1788591,6953,257.24
AIR EUROPA LINEAS AEREAS SOCIEDAD ANONIMA,3824484,15087,253.5
EDELWEISS AIR AG,197258,800,246.57
IBÉRIA LINEAS AEREAS DE ESPAÑA SOCIEDAD ANONIMA OPERADORA,9727150,39521,246.13
IBERWORLD AIRLINES S.A.,29748,126,236.1


In [0]:
-- Pergunta 4: Quais rotas apresentaram as maiores distâncias entre os aeródromos de origem e destino?

SELECT
    AEROPORTO_ORIGEM_NOME,
    AEROPORTO_ORIGEM_PAIS,
    AEROPORTO_DESTINO_NOME,
    AEROPORTO_DESTINO_PAIS,
    MAX(DISTANCIA_VOADA_KM) AS distancia_voada_km,
    SUM(DECOLAGENS) AS quantidade_decolagens
FROM workspace.default.vw_voos_completos
WHERE DISTANCIA_VOADA_KM IS NOT NULL
  AND AEROPORTO_ORIGEM_NOME IS NOT NULL
  AND AEROPORTO_DESTINO_NOME IS NOT NULL
GROUP BY
    AEROPORTO_ORIGEM_NOME,
    AEROPORTO_ORIGEM_PAIS,
    AEROPORTO_DESTINO_NOME,
    AEROPORTO_DESTINO_PAIS
ORDER BY distancia_voada_km DESC
LIMIT 10;

AEROPORTO_ORIGEM_NOME,AEROPORTO_ORIGEM_PAIS,AEROPORTO_DESTINO_NOME,AEROPORTO_DESTINO_PAIS,distancia_voada_km,quantidade_decolagens
GUARULHOS,BRASIL,DOHA,QATAR,1043504,7839
DOHA,QATAR,GUARULHOS,BRASIL,1019788,7814
GUARULHOS,BRASIL,PARIS,FRANÇA,1015740,29105
PARIS,FRANÇA,GUARULHOS,BRASIL,1015740,28966
PANAMA,PANAMÁ,GUARULHOS,BRASIL,951082,29754
GUARULHOS,BRASIL,PANAMA,PANAMÁ,945996,29814
GUARULHOS,BRASIL,"MIAMI, FLORIDA",ESTADOS UNIDOS DA AMÉRICA,815176,45669
"MIAMI, FLORIDA",ESTADOS UNIDOS DA AMÉRICA,GUARULHOS,BRASIL,808602,48406
GUARULHOS,BRASIL,LISBOA,PORTUGAL,769695,20976
LISBOA,PORTUGAL,GUARULHOS,BRASIL,769695,20839


In [0]:
-- Pergunta 5: Quais etapas de voo apresentaram as maiores quantidades de passageiros transportados?

SELECT
    VOO_ID,
    MES_ANO,
    EMPRESA_NOME,
    EMPRESA_NACIONALIDADE,
    AEROPORTO_ORIGEM_NOME,
    AEROPORTO_DESTINO_NOME,
    NATUREZA,
    GRUPO_DE_VOO,
    COALESCE(PASSAGEIROS_PAGOS, 0) + COALESCE(PASSAGEIROS_GRATIS, 0) AS total_passageiros,
    PASSAGEIROS_PAGOS,
    PASSAGEIROS_GRATIS,
    DISTANCIA_VOADA_KM,
    DECOLAGENS,
    ROUND(DISTANCIA_VOADA_KM / NULLIF(DECOLAGENS, 0), 1) AS distancia_por_decolagem_km,
    ROUND((COALESCE(PASSAGEIROS_PAGOS, 0) + COALESCE(PASSAGEIROS_GRATIS, 0)) / NULLIF(DECOLAGENS, 0), 1) AS passageiros_por_decolagem,
    ASSENTOS
FROM workspace.default.vw_voos_completos
ORDER BY total_passageiros DESC
LIMIT 10;

VOO_ID,MES_ANO,EMPRESA_NOME,EMPRESA_NACIONALIDADE,AEROPORTO_ORIGEM_NOME,AEROPORTO_DESTINO_NOME,NATUREZA,GRUPO_DE_VOO,total_passageiros,PASSAGEIROS_PAGOS,PASSAGEIROS_GRATIS,DISTANCIA_VOADA_KM,DECOLAGENS,distancia_por_decolagem_km,passageiros_por_decolagem,ASSENTOS
850691,12/2019,TAM LINHAS AÉREAS S.A.,BRASILEIRA,SÃO PAULO,RIO DE JANEIRO,DOMÉSTICA,REGULAR,93719,92357,1362,256566,701,366.0,133.7,107394
845077,01/2019,TAM LINHAS AÉREAS S.A.,BRASILEIRA,RIO DE JANEIRO,SÃO PAULO,DOMÉSTICA,REGULAR,93347,91980,1367,262788,718,366.0,130.0,103392
849617,10/2019,TAM LINHAS AÉREAS S.A.,BRASILEIRA,RIO DE JANEIRO,SÃO PAULO,DOMÉSTICA,REGULAR,92828,91760,1068,267546,731,366.0,127.0,105414
851089,01/2020,TAM LINHAS AÉREAS S.A.,BRASILEIRA,RIO DE JANEIRO,SÃO PAULO,DOMÉSTICA,REGULAR,91898,90450,1448,253638,693,366.0,132.6,106470
364265,09/2014,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.),BRASILEIRA,RIO DE JANEIRO,SÃO PAULO,DOMÉSTICA,REGULAR,91515,88965,2550,294630,805,366.0,113.7,142407
406532,12/2019,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.),BRASILEIRA,SÃO PAULO,RIO DE JANEIRO,DOMÉSTICA,REGULAR,90239,88017,2222,255834,699,366.0,129.1,124878
844657,12/2018,TAM LINHAS AÉREAS S.A.,BRASILEIRA,SÃO PAULO,RIO DE JANEIRO,DOMÉSTICA,REGULAR,89804,88386,1418,257664,704,366.0,127.6,101376
364401,09/2014,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.),BRASILEIRA,SÃO PAULO,RIO DE JANEIRO,DOMÉSTICA,REGULAR,88861,86134,2727,294264,804,366.0,110.5,142191
849702,10/2019,TAM LINHAS AÉREAS S.A.,BRASILEIRA,SÃO PAULO,RIO DE JANEIRO,DOMÉSTICA,REGULAR,88398,87606,792,269742,737,366.0,119.9,106278
366834,12/2014,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.),BRASILEIRA,SÃO PAULO,RIO DE JANEIRO,DOMÉSTICA,REGULAR,88375,85788,2587,272670,745,366.0,118.6,125157
